# Governed YOLO Training Pipeline

Thin Jupyter/Kaggle interface over `edge_ai_mass.training`. ZenML orders preprocessing, tuning, training, evaluation, export, registration, and reusable-output publication; the package owns the implementation while this notebook only supplies environment-specific paths and chooses stages.

Pipeline: **TACO + AquaTrash + RealWaste COCO segmentations → versioned YOLO dataset → Optuna → YOLO → MLflow registry → Kaggle training-output dataset**.

In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

IS_KAGGLE = Path('/kaggle').exists()
INPUT_ROOT = Path('/kaggle/input') if IS_KAGGLE else Path('data').resolve()
WORK_ROOT = Path('/kaggle/working') if IS_KAGGLE else Path('.').resolve()
print({'is_kaggle': IS_KAGGLE, 'input_root': str(INPUT_ROOT), 'work_root': str(WORK_ROOT)})

# Install only dependencies missing from the Kaggle image. The kernel metadata
# enables internet so a pushed version can run without manual setup cells.
REQUIRED_PACKAGES = {
    'yaml': 'pyyaml>=6.0',
    'PIL': 'pillow>=10.0',
    'ultralytics': 'ultralytics>=8.4',
    'mlflow': 'mlflow>=3.0',
    'optuna': 'optuna>=4.0',
    'zenml': 'zenml>=0.95,<0.96',
    'kaggle': 'kaggle>=2.2.2',
    'onnx': 'onnx>=1.16',
    'plotly': 'plotly>=5.0',
}
missing_packages = [spec for module, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing_packages])

In [ ]:
!pip install SQLAlchemy-Utils sqlmodel passlib pymysql
!pip uninstall onnxruntime-gpu
!pip install -q onnxruntime


In [ ]:
!zenml init

## Load the project module

On Kaggle, upload this repository (or a wheel) as a dataset. The cell finds a checkout containing `src/edge_ai_mass`; set `EDGE_AI_MASS_MODULE_ROOT` if auto-discovery is ambiguous.

In [ ]:
!rm -rf edge-ai-mass-estimation
!git clone https://github.com/karimaouaouda/edge-ai-mass-estimation.git -b dev --depth=1

REPO_PATH = Path("/kaggle/working/edge-ai-mass-estimation")
MODULE_PATH = REPO_PATH / "src"

sys.path.insert(1, str(MODULE_PATH))

import edge_ai_mass

print(f"Completed cloned and added the script to the working space and system path, u can use it now")

## Bind mounted datasets

TACO is acquired into `/kaggle/working` from the COCO `annotations.json` hosted on Hugging Face; its image URLs are downloaded concurrently and resumably. AquaTrash remains mounted. RealWaste uses a mounted raw `realwaste-main/RealWaste/<original-class>` tree plus a separate COCO segmentation JSON; JSON categories provide project labels and original folder classes are ignored.

In [ ]:
# Acquire TACO from the Hugging Face COCO JSON used by the previous Kaggle
# notebook. Existing valid files are reused when a kernel cell is rerun.
if IS_KAGGLE:
    from edge_ai_mass.training import TACO_ANNOTATIONS_URL, download_taco_dataset

    taco_root = WORK_ROOT / 'data' / 'raw' / 'taco'
    taco_summary = download_taco_dataset(
        taco_root,
        annotations_url=os.getenv('TACO_ANNOTATIONS_URL', TACO_ANNOTATIONS_URL),
        max_workers=int(os.getenv('TACO_DOWNLOAD_WORKERS', '8')),
        timeout_seconds=float(os.getenv('TACO_DOWNLOAD_TIMEOUT', '30')),
        retries=int(os.getenv('TACO_DOWNLOAD_RETRIES', '3')),
        max_failures=int(os.getenv('TACO_MAX_DOWNLOAD_FAILURES', '0')),
    )
    os.environ['TACO_ANNOTATIONS'] = str(taco_root / 'annotations.json')
    os.environ['TACO_IMAGES'] = str(taco_root)

    # Bind AquaTrash mounts directly: its raw images and segmentation JSON
    # are published as separate Kaggle datasets.
    os.environ.setdefault('AQUATRASH_ANNOTATIONS', "/kaggle/input/datasets/karimaouaouda/aquatrash-segmantations/labels_final.json")
    os.environ.setdefault('AQUATRASH_IMAGES',"/kaggle/input/datasets/harshpanwar/aquatrash/Images")


    print(f"os env : {os.environ.get("AQUATRASH_IMAGES")}")
    print(f"os env : {os.environ.get("AQUATRASH_ANNOTATIONS")}")
    if Path(os.environ.get("AQUATRASH_ANNOTATIONS")).exists():
        print(f"annotations exists")
    if Path(os.environ.get("AQUATRASH_IMAGES")).exists():
        print(f"annotations exists")
    else:
        print(f"folder not found")

    # RealWaste uses two independent inputs: a COCO segmentation JSON whose
    # categories are our eight labels, and the untouched raw dataset whose
    # class-folder names are unrelated to our taxonomy.
    from edge_ai_mass.training.taxonomy import WASTE_CLASSES

    def discover_realwaste_annotations() -> Path:
        override = os.getenv('REALWASTE_ANNOTATIONS', "/kaggle/input/datasets/karimaouaouda/realwaste-segmentations/instances_default.json")
        candidates = [Path(override)] if override else []
        candidates += sorted(INPUT_ROOT.rglob('annotations.json'))
        accepted = []
        for candidate in dict.fromkeys(path.resolve() for path in candidates):
            if not candidate.is_file():
                continue
            try:
                payload = json.loads(candidate.read_text(encoding='utf-8'))
                category_names = {str(item['name']).strip() for item in payload.get('categories', [])}
                has_segments = any(item.get('segmentation') for item in payload.get('annotations', []))
            except (OSError, ValueError, KeyError, TypeError):
                continue
            # Reject TACO/AquaTrash and legacy one-class `trash` JSON files.
            if category_names and category_names <= set(WASTE_CLASSES) and has_segments:
                path_hint = 1 if 'realwaste' in candidate.as_posix().casefold() else 0
                accepted.append((path_hint, len(payload.get('annotations', [])), candidate))
        if not accepted:
            raise FileNotFoundError(
                'No RealWaste COCO segmentation JSON with project labels was found. '
                'Attach it or set REALWASTE_ANNOTATIONS.'
            )
        accepted.sort(key=lambda item: (item[0], item[1], item[2].as_posix()), reverse=True)
        return accepted[0][2]

    def discover_realwaste_images() -> Path:
        override = os.getenv('REALWASTE_IMAGES', "/kaggle/input/datasets/joebeachcapital/realwaste/realwaste-main/RealWaste")
        candidates = [Path(override)] if override else []
        # joebeachcapital/realwaste mounts with this exact nested layout.
        candidates += sorted(INPUT_ROOT.glob('*/realwaste-main/RealWaste'))
        for candidate in dict.fromkeys(path.resolve() for path in candidates):
            if candidate.is_dir() and any(path.is_dir() for path in candidate.iterdir()):
                return candidate
        raise FileNotFoundError(
            'Could not find realwaste-main/RealWaste. Attach the raw RealWaste dataset '
            'or set REALWASTE_IMAGES to that directory.'
        )

    realwaste_annotations = discover_realwaste_annotations()
    realwaste_images = discover_realwaste_images()
    os.environ['REALWASTE_ANNOTATIONS'] = str(realwaste_annotations)
    os.environ['REALWASTE_IMAGES'] = str(realwaste_images)
    print({'realwaste_annotations': str(realwaste_annotations), 'realwaste_images': str(realwaste_images)})

OVERRIDES = []
if IS_KAGGLE:
     OVERRIDES += [
        'data.output_dir=/kaggle/working/data/processed/waste_seg_yolo',
        'data.materialize=copy',
        'training.artifacts_dir=/kaggle/working/artifacts/training/yolo',
        'tuning.storage=/kaggle/working/artifacts/optuna/yolo.db',
        'tracking.uri=sqlite:////kaggle/working/artifacts/mlflow/mlflow.db',
        'tracking.registry_uri=sqlite:////kaggle/working/artifacts/mlflow/mlflow.db',
        'training.epochs=60',
         'training.checkpointing.resume.selected_epoch=20',
         'training.early_stoping.patience=10',
        'training.device=cpu',
        'training.workers=2',
        'train.batch=2'
        'training.additional_epochs=0',
        'evaluation.precision=fp16',
        'evaluation.imgsz=512',
        'evaluation.device=cpu',
        'evaluation.batch=1',
        'evaluation.plots=true',
        f"training.checkpointing.interval_epochs={int(os.getenv('TRAIN_CHECKPOINT_INTERVAL', '5'))}",
        f"training.checkpointing.resume.mode={os.getenv('TRAIN_RESUME_MODE', 'auto')}",
        f"training.checkpointing.resume.additional_epochs={int(os.getenv('TRAIN_ADDITIONAL_EPOCHS', '0'))}",
        'publication.bundle_dir=/kaggle/working/publication/training-outputs',
        f"publication.dataset={os.getenv('TRAINING_OUTPUT_DATASET', 'aouaoudakarim/edge-ai-mass-training-outputs')}",
        'tuning.trials=5',
        'tuning.epochs_per_trial=8',
        'tuning.search_space.batch.choices=[2, 4, 8]',
        'tuning.search_space.imgsz.choices=[512, 640]',
    ]
EXPORT_FORMATS = [value.strip() for value in os.getenv('TRAIN_EXPORT_FORMATS', '').split(',') if value.strip()]
if EXPORT_FORMATS:
    OVERRIDES += ['export.enabled=true', f"export.formats=[{', '.join(EXPORT_FORMATS)}]"]
print('Overrides:', OVERRIDES)
print(f"from updated one")

In [ ]:
from edge_ai_mass.training import TrainingPipeline, restore_training_outputs

# If a previous training-output dataset is attached, restore checkpoints and
# model state before constructing the pipeline. Raw/processed data is never
# present in this archive and must still come from the normal data sources.
if IS_KAGGLE:
    output_archive_override = os.getenv('TRAINING_OUTPUT_ARCHIVE', "/kaggle/input/datasets/abdouaoualmia/yolo-training-output/artifacts/training/yolo/waste-seg-yolo")
    output_archives = [Path(output_archive_override)] if output_archive_override else []
    output_archives += sorted(INPUT_ROOT.glob('*/training_outputs.zip'))
    output_archive = next((path for path in output_archives if path.is_file()), None)
    if output_archive_override is not None:
        restored = restore_training_outputs(
            output_archive_override,
            WORK_ROOT / 'artifacts' / 'training' / 'yolo' / 'waste-seg-yolo',
            overwrite=False,
        )
        print('Restored reusable training outputs:', restored)

CONFIG_PATH = "/kaggle/working/edge-ai-mass-estimation/configs/training/yolo_segmentation_pretrain.yaml"
pipeline = TrainingPipeline.from_config(CONFIG_PATH, overrides=OVERRIDES)
plan = pipeline.plan('all')
print(json.dumps(plan, indent=2))

missing = [item['name'] for item in plan['sources'] if item['required'] and not (item['annotations_exist'] and item['images_exist'])]
if missing:
    print('Required mounts still missing:', missing)

## Execute

Use `preprocess` first while validating mounts and dataset mosaics, then resume with `tune,train,evaluate,export,register,publish`. ZenML records the complete ordered DAG. Set `TRAIN_EXPORT_FORMATS=onnx,engine` to request one or multiple best-model exports. Final training writes periodic resumable snapshots under `artifacts/.../checkpoints`; `TRAIN_CHECKPOINT_INTERVAL`, `TRAIN_RESUME_MODE`, `TRAINING_RESUME_CHECKPOINT`, and `TRAIN_ADDITIONAL_EPOCHS` control checkpoint loading and chunk size. The final publish step versions a reusable-output Kaggle dataset without raw or processed data.

In [ ]:
import torch
!export PYTORCH_ALLOC_CONF=expandable_segments:True
import gc


gc.collect()
torch.cuda.empty_cache()

num_gpus = torch.cuda.device_count()

print(f"there is {num_gpus} gpu can use")

for i in range(torch.cuda.device_count()):
    print(f"Device Number [{i}]: {torch.cuda.get_device_name(i)}")

os.environ['CUDA_VISIBLE_DEVICES']: 0

In [ ]:
!rm -rf edge-ai-mass-estimation
!git clone https://github.com/karimaouaouda/edge-ai-mass-estimation.git -b dev --depth=1

!pip uninstall -y edge-ai-mass edge_ai_mass || true


!cd kaggle/working/edge-ai-mass-estimation && pip install -e .


REPO_PATH = Path("/kaggle/working/edge-ai-mass-estimation")
MODULE_PATH = REPO_PATH / "src"

sys.path.insert(1, str(MODULE_PATH))

import importlib

import edge_ai_mass
import inspect

importlib.reload(edge_ai_mass)

print(edge_ai_mass.__file__)
print(inspect.getfile(edge_ai_mass))


print(f"Completed cloned and added the script to the working space and system path, u can use it now")

In [ ]:
!export PYTORCH_ALLOC_CONF=expandable_segments:True
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.multiprocessing.freeze_support()
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1,0")
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("CUDA available =", torch.cuda.is_available())
print("CUDA device count =", torch.cuda.device_count())

for i, s in enumerate(OVERRIDES):
    if s == "evaluation.batch=8":
        OVERRIDES[i] = "evaluation.batch=2"
    if s == "evaluation.plots=true":
        OVERRIDES[i] = "evaluation.plots=false"

for i, s in enumerate(OVERRIDES):
    print(f"- {s}")


In [ ]:
# A pushed Kaggle version runs the complete pipeline by default. Change
# TRAIN_STAGES while editing interactively to resume selected stages.

RUN_STAGES = "all"
SKIP_OPTUNA = True
print({'run_stages': RUN_STAGES, 'skip_optuna': SKIP_OPTUNA})
results = pipeline.run(RUN_STAGES, skip_optuna=SKIP_OPTUNA, force=True)
print(json.dumps(results, indent=2, default=str)[:20000])

In [ ]:
# Inspect durable outputs after any stage.
state_path = pipeline.config.artifacts_dir / 'pipeline_state.json'
if state_path.exists():
    print(state_path.read_text())
print('Dataset manifest:', pipeline.config.dataset_dir / 'dataset_manifest.json')
print('Training artifacts:', pipeline.config.artifacts_dir)